Inicio Setup

In [ ]:
# ============================================================
# Configuração inicial do ambiente e caminhos do projeto
# Objetivo:
#   Importar bibliotecas necessárias e definir o caminho base
#   para acesso aos arquivos da camada raw/bronze.
# ============================================================

# 1) Importação de bibliotecas
# ------------------------------------------------------------
# pandas: manipulação e análise de dados
# pathlib.Path: manipulação de caminhos de arquivos de forma
#               mais segura e independente do sistema operacional

import pandas as pd
from pathlib import Path


# 2) Definição do caminho base dos dados
# ------------------------------------------------------------
# BASE_PATH representa o diretório raiz onde os dados brutos
# (camada raw/bronze) estão armazenados.
#
# Boa prática:
# - Utilizar Path em vez de strings com barras
# - Permite encadeamento com / para formar novos caminhos

BASE_PATH = Path("../data/raw_bronze")


# Exemplo de uso recomendado:
# arquivo = BASE_PATH / "arquivo.csv"
# df = pd.read_csv(arquivo)

Leitura dos dados

In [ ]:
# ============================================================
# Leitura dos arquivos da camada raw/bronze
# Objetivo:
#   Carregar os datasets brutos para DataFrames do pandas
#   a partir do caminho base previamente definido.
# ============================================================

# 1) Leitura do arquivo de bairros
# ------------------------------------------------------------
# Contém informações geográficas ou administrativas dos bairros.

df_bairros = pd.read_csv(BASE_PATH / "bairros.csv")


# 2) Leitura do arquivo de concorrentes
# ------------------------------------------------------------
# Contém informações sobre estabelecimentos concorrentes.

df_conc = pd.read_csv(BASE_PATH / "concorrentes.csv")


# 3) Leitura do arquivo de eventos de fluxo
# ------------------------------------------------------------

df_eventos = pd.read_csv(BASE_PATH / "eventos_de_fluxo.csv")


# 4) Leitura do arquivo de população
# ------------------------------------------------------------
# Arquivo em formato JSON.
# Pode conter estrutura aninhada; se necessário, utilizar
# pd.json_normalize() após a leitura.

df_pop = pd.read_json(BASE_PATH / "populacao.json")



INICIO DA EXPLORAÇÃO

Análise Exploratória Inicial dos Dados

Nesta etapa foi realizada uma análise exploratória dos DataFrames com o objetivo de compreender sua estrutura, qualidade e consistência.

Foram avaliados os seguintes pontos:

Dimensão do dataset (Shape) – verificação da quantidade de linhas e colunas.

Estrutura das colunas (Info) – análise dos tipos de dados, contagem de valores não nulos e consumo de memória.

Valores nulos – identificação de possíveis problemas de completude.
O grau em que os dados estão preenchidos, sem valores ausentes (nulos), conforme esperado.

Registros duplicados – verificação de redundância nos dados.

Quantidade de linhas – análise da quantidade total de linhas.

Cardinalidade de chaves – verificação da quantidade de valores únicos na coluna identificadora.

Essa análise permite identificar possíveis inconsistências, problemas de tipagem e necessidade de tratamento antes das etapas de transformação ou modelagem.

BAIRROS

In [8]:
# ============================================================
# Análise exploratória inicial - DataFrame df_bairros
# Objetivo:
#   Avaliar estrutura, qualidade e integridade dos dados
#   através de métricas essenciais de validação.
# ============================================================

# 1) Shape do DataFrame
# ------------------------------------------------------------
# Exibe quantidade de linhas e colunas.
# Permite avaliar o volume de dados disponível.

print(" SHAPE ".center(70, "="))
print(f"Shape: {df_bairros.shape}")


# 2) Informações estruturais
# ------------------------------------------------------------
# Exibe:
# - Tipos de dados
# - Quantidade de valores não nulos
# - Uso de memória
# Essencial para validação de tipagem e possíveis ajustes de schema.

print(" INFO ".center(70, "="))
df_bairros.info()


# 3) Análise de valores nulos
# ------------------------------------------------------------
# Calcula:
# - Total de valores nulos por coluna
# - Percentual de nulos
# - Percentual de completude
#
# Boa prática:
# - Trabalhar com percentual para melhor comparação entre colunas
# - Avaliar impacto de nulos em colunas críticas

print(" NULOS ".center(70, "="))

nulls = df_bairros.isnull().sum()
percent_nulls = (df_bairros.isnull().mean() * 100).round(2)
completude = 100 - percent_nulls

display(
    pd.DataFrame({
        "Nulos": nulls,
        "% Nulos": percent_nulls,
        "% Completude": completude
    })
)


# 4) Verificação de registros duplicados
# ------------------------------------------------------------
# Avalia duplicidade total do DataFrame.
# Em cenários reais, recomenda-se validar duplicidade por chave primária.

print(" DUPLICADOS ".center(70, "="))
print(f"Duplicados: {df_bairros.duplicated().sum()}")


# 5) Quantidade total de registros
# ------------------------------------------------------------
# Exibe total de linhas formatado com separador de milhar.

print(" QUANTIDADE DE REGISTROS ".center(70, "="))
print(f"Total de linhas: {df_bairros.shape[0]:,}")


# 6) Validação de unicidade da chave
# ------------------------------------------------------------
# Verifica a quantidade de valores únicos na coluna 'codigo'.
# Boa prática:
# - Confirmar se a chave possui 100% de unicidade
# - Se nunique() < total de linhas, investigar possíveis inconsistências

print(" CHAVES ÚNICAS ".center(70, "="))
print(f"Chaves únicas: {df_bairros['codigo'].nunique()}")

=============================== SHAPE ================================
Shape: (133, 5)
================================ INFO ================================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   codigo     133 non-null    int64  
 1   nome       133 non-null    object 
 2   municipio  133 non-null    object 
 3   uf         133 non-null    object 
 4   area       133 non-null    float64
dtypes: float64(1), int64(1), object(3)
memory usage: 5.3+ KB
=============================== NULOS ================================


,Nulos,% Nulos,% Completude
codigo,0,0.0,100.0
nome,0,0.0,100.0
municipio,0,0.0,100.0
uf,0,0.0,100.0
area,0,0.0,100.0


============================= DUPLICADOS =============================
Duplicados: 0
====================== QUANTIDADE DE REGISTROS =======================
Total de linhas: 133
=========================== CHAVES ÚNICAS ============================
Chaves únicas: 133


CONCORRENTES

In [10]:
# ============================================================
# Análise exploratória inicial - DataFrame df_conc
# Objetivo:
#   Avaliar estrutura, qualidade, integridade e variabilidade
#   dos dados de concorrentes.
# ============================================================

# 1) Shape do DataFrame
# ------------------------------------------------------------
# Exibe quantidade de linhas e colunas.
# Permite entender o volume total da base.

print(" SHAPE ".center(70, "="))
print(f"Shape: {df_conc.shape}")


# 2) Informações estruturais
# ------------------------------------------------------------
# Exibe:
# - Tipos de dados
# - Quantidade de valores não nulos
# - Uso de memória
# Fundamental para validar schema e possíveis ajustes de tipagem.

print(" INFO ".center(70, "="))
df_conc.info()


# 3) Análise de completude
# ------------------------------------------------------------
# Calcula:
# - Total de valores nulos por coluna
# - Percentual de nulos
# - Percentual de completude
#
# Boa prática:
# - Avaliar percentual para comparação proporcional entre colunas
# - Identificar colunas críticas com baixa completude

print(" COMPLETUDE ".center(70, "="))

nulls = df_conc.isnull().sum()
percent_nulls = (df_conc.isnull().mean() * 100).round(2)
completude = 100 - percent_nulls

display(pd.DataFrame({
    "Nulos": nulls,
    "% Nulos": percent_nulls,
    "% Completude": completude
}))


# 4) Verificação de duplicidade
# ------------------------------------------------------------
# Avalia registros totalmente duplicados.
# Em análises mais robustas, recomenda-se verificar duplicidade
# com base em possíveis chaves naturais.

print(" DUPLICADOS ".center(70, "="))
print(f"Duplicados: {df_conc.duplicated().sum()}")


# 5) Quantidade total de registros
# ------------------------------------------------------------
# Exibe total de linhas formatado com separador de milhar.

print(" QUANTIDADE DE REGISTROS ".center(70, "="))
print(f"Total de linhas: {df_conc.shape[0]:,}")


# 6) Cardinalidade das colunas
# ------------------------------------------------------------
# Exibe quantidade de valores únicos por coluna.
# Útil para:
# - Identificar possíveis chaves
# - Detectar colunas categóricas
# - Avaliar variabilidade dos dados

print(" CARDINALIDADE ".center(70, "="))
display(df_conc.nunique())

=============================== SHAPE ================================
Shape: (4202, 8)
================================ INFO ================================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4202 entries, 0 to 4201
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   codigo         4202 non-null   int64  
 1   nome           4202 non-null   object 
 2   categoria      4202 non-null   object 
 3   faixa_preco    4202 non-null   int64  
 4   endereco       4199 non-null   object 
 5   municipio      4198 non-null   object 
 6   uf             4198 non-null   object 
 7   codigo_bairro  1450 non-null   float64
dtypes: float64(1), int64(2), object(5)
memory usage: 262.8+ KB
============================= COMPLETUDE =============================


,Nulos,% Nulos,% Completude
codigo,0,0.00,100.00
nome,0,0.00,100.00
categoria,0,0.00,100.00
faixa_preco,0,0.00,100.00
endereco,3,0.07,99.93
municipio,4,0.10,99.90
uf,4,0.10,99.90
codigo_bairro,2752,65.49,34.51


============================= DUPLICADOS =============================
Duplicados: 0
====================== QUANTIDADE DE REGISTROS =======================
Total de linhas: 4,202
=========================== CARDINALIDADE ============================


codigo           4202
nome             4161
categoria        1786
faixa_preco         5
endereco         4166
municipio          55
uf                  1
codigo_bairro     106
dtype: int64

EVENTOS

In [ ]:
# ============================================================
# Análise exploratória inicial - DataFrame df_eventos
# Objetivo:
#   Avaliar estrutura, qualidade, integridade e variabilidade
#   dos dados de eventos de fluxo.
# ============================================================

# 1) Shape do DataFrame
# ------------------------------------------------------------
# Exibe quantidade de linhas e colunas.
# Permite avaliar o volume total de registros.

print(" SHAPE ".center(70, "="))
print(f"Shape: {df_eventos.shape}")


# 2) Informações estruturais
# ------------------------------------------------------------
# Exibe:
# - Tipos de dados
# - Quantidade de valores não nulos por coluna
# - Uso de memória
# Importante para validar tipagem (datas, numéricos, categorias).

print(" INFO ".center(70, "="))
df_eventos.info()


# 3) Análise de completude
# ------------------------------------------------------------
# Calcula:
# - Total de valores nulos por coluna
# - Percentual de nulos
# - Percentual de completude
#
# Boa prática:
# - Trabalhar com percentual para análise proporcional
# - Identificar colunas críticas para modelagem

print(" COMPLETUDE ".center(70, "="))

nulls = df_eventos.isnull().sum()
percent_nulls = (df_eventos.isnull().mean() * 100).round(2)
completude = 100 - percent_nulls

display(pd.DataFrame({
    "Nulos": nulls,
    "% Nulos": percent_nulls,
    "% Completude": completude
}))


# 4) Verificação de duplicidade
# ------------------------------------------------------------
# Avalia registros totalmente duplicados.
# Em cenários de eventos, é recomendável validar duplicidade

print(" DUPLICADOS ".center(70, "="))
print(f"Duplicados: {df_eventos.duplicated().sum()}")


# 5) Quantidade total de registros
# ------------------------------------------------------------
# Exibe total de linhas com formatação para melhor leitura.

print(" QUANTIDADE DE REGISTROS ".center(70, "="))
print(f"Total de linhas: {df_eventos.shape[0]:,}")


# 6) Cardinalidade das colunas
# ------------------------------------------------------------
# Exibe quantidade de valores únicos por coluna.
# Útil para:
# - Detectar possíveis identificadores únicos
# - Identificar colunas categóricas
# - Avaliar variabilidade dos dados

print(" CARDINALIDADE ".center(70, "="))
display(df_eventos.nunique())

=============================== SHAPE ================================
Shape: (248589, 3)
================================ INFO ================================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248589 entries, 0 to 248588
Data columns (total 3 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   codigo              248589 non-null  object
 1   datetime            248589 non-null  object
 2   codigo_concorrente  248589 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 5.7+ MB
============================= COMPLETUDE =============================


,Nulos,% Nulos,% Completude
codigo,0,0.0,100.0
datetime,0,0.0,100.0
codigo_concorrente,0,0.0,100.0


============================= DUPLICADOS =============================
Duplicados: 38788
====================== QUANTIDADE DE REGISTROS =======================
Total de linhas: 248,589
=========================== CARDINALIDADE ============================


codigo                 28036
datetime              209623
codigo_concorrente       664
dtype: int64

POPULACAO

In [14]:
# ============================================================
# Análise exploratória inicial - DataFrame df_pop
# Objetivo:
#   Avaliar estrutura, qualidade, integridade e variabilidade
#   dos dados de população.
# ============================================================

# 1) Shape do DataFrame
# ------------------------------------------------------------
# Exibe quantidade de linhas e colunas.
# Permite avaliar rapidamente o volume da base.

print(" SHAPE ".center(70, "="))
print(f"Shape: {df_pop.shape}")


# 2) Informações estruturais
# ------------------------------------------------------------
# Exibe:
# - Tipos de dados
# - Quantidade de valores não nulos
# - Uso de memória
# Importante especialmente para JSON, que pode gerar colunas
# com tipos inadequados ou estruturas aninhadas.

print(" INFO ".center(70, "="))
df_pop.info()


# 3) Análise de completude
# ------------------------------------------------------------
# Calcula:
# - Total de valores nulos por coluna
# - Percentual de nulos
# - Percentual de completude
#
# Boa prática:
# - Trabalhar com percentual para comparação proporcional
# - Identificar colunas com necessidade de tratamento

print(" COMPLETUDE ".center(70, "="))

nulls = df_pop.isnull().sum()
percent_nulls = (df_pop.isnull().mean() * 100).round(2)
completude = 100 - percent_nulls

display(pd.DataFrame({
    "Nulos": nulls,
    "% Nulos": percent_nulls,
    "% Completude": completude
}))


# 4) Verificação de duplicidade
# ------------------------------------------------------------
# Avalia registros totalmente duplicados.
# Em bases populacionais, pode ser interessante validar
# duplicidade por chave geográfica (ex: código do bairro).

print(" DUPLICADOS ".center(70, "="))
print(f"Duplicados: {df_pop.duplicated().sum()}")


# 5) Quantidade total de registros
# ------------------------------------------------------------
# Exibe total de linhas com formatação para melhor leitura.

print(" QUANTIDADE DE REGISTROS ".center(70, "="))
print(f"Total de linhas: {df_pop.shape[0]:,}")


# 6) Cardinalidade das colunas
# ------------------------------------------------------------
# Exibe quantidade de valores únicos por coluna.
# Útil para:
# - Identificar possíveis chaves
# - Avaliar variabilidade
# - Detectar colunas categóricas

print(" CARDINALIDADE ".center(70, "="))
display(df_pop.nunique())

=============================== SHAPE ================================
Shape: (133, 2)
================================ INFO ================================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133 entries, 0 to 132
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   codigo     133 non-null    int64  
 1   populacao  132 non-null    float64
dtypes: float64(1), int64(1)
memory usage: 2.2 KB
============================= COMPLETUDE =============================


,Nulos,% Nulos,% Completude
codigo,0,0.00,100.00
populacao,1,0.75,99.25


============================= DUPLICADOS =============================
Duplicados: 0
====================== QUANTIDADE DE REGISTROS =======================
Total de linhas: 133
=========================== CARDINALIDADE ============================


codigo       133
populacao    132
dtype: int64

ANÁLISE DE DUPLICIDADE – EVENTOS

Foi identificado um volume significativo de registros duplicados na tabela de eventos (aproximadamente 65,49%).

Para garantir que se tratam de duplicidades completas (mesmo cliente, mesmo concorrente e mesmo timestamp), foi realizada uma verificação utilizando duplicated(keep=False) ordenando por código e datetime.

A análise confirmou a existência de registros idênticos, caracterizando duplicidade técnica do dataset.

In [ ]:
# ============================================================
# Análise de registros duplicados - df_eventos
# Objetivo:
#   Identificar e visualizar registros duplicados considerando
#   todas as colunas do DataFrame.
# ============================================================

# 1) Identificação de duplicados
# ------------------------------------------------------------
# duplicated(keep=False):
#   - Diferente do padrão (keep='first'), que mantém apenas a primeira
#
# Boa prática:
# - Usar keep=False para análise diagnóstica completa
# - Permite visualizar todos os registros envolvidos na duplicidade

duplicados_eventos = df_eventos[df_eventos.duplicated(keep=False)]


# 2) Ordenação para análise
# ------------------------------------------------------------
# Ordena por:
#   - codigo (possível identificador da entidade)
#   - datetime (ordenação temporal)
#
# Facilita:
# - Identificar padrões de repetição
# - Verificar se duplicidade é erro ou evento legítimo repetido

duplicados_eventos = duplicados_eventos.sort_values(
    ["codigo", "datetime"]
)


# 3) Visualização inicial
# ------------------------------------------------------------
# head(10):
#   - Exibe os 10 primeiros registros para inspeção rápida
#   - Evita sobrecarregar a visualização em bases grandes

duplicados_eventos.head(10)

,codigo,datetime,codigo_concorrente
181487,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 12:47:25.353,881829878586104
181496,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 12:47:25.353,881829878586104
181950,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 13:37:25.502,881829878586104
181986,++GYUHjNItx16mLsDMbXPa9VpiP6TjhqjiCnDYhDuMDPUJ...,2017-07-12 13:37:25.502,881829878586104
50179,++TPZAjdD0bRDGncsLVDQ2xWxyMkxZvWRBKw7Nr90nIATL...,2017-07-01 11:39:50.926,283641161989443
50180,++TPZAjdD0bRDGncsLVDQ2xWxyMkxZvWRBKw7Nr90nIATL...,2017-07-01 11:39:50.926,283641161989443
50172,++TPZAjdD0bRDGncsLVDQ2xWxyMkxZvWRBKw7Nr90nIATL...,2017-07-11 08:43:46.777,283641161989443
50202,++TPZAjdD0bRDGncsLVDQ2xWxyMkxZvWRBKw7Nr90nIATL...,2017-07-11 08:43:46.777,283641161989443
205769,++d/SQrkpJ4Adgaj3i0mIjPbotMS+b76Yz1d97SWbdSMIs...,2017-07-24 09:58:03.628,211569545688921
205777,++d/SQrkpJ4Adgaj3i0mIjPbotMS+b76Yz1d97SWbdSMIs...,2017-07-24 09:58:03.628,211569545688921
